# 01 — Cleaning & EDA

Online Retail II → cleaned gold facts used by the CommercePulse dashboard.

**Checks**
- Revenue / orders / customers / AOV vs dashboard cards (~£20.1M, ~39.6K, ~5.9K)
- Monthly seasonality (November peaks)
- Guest vs registered mix and return rate


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..')
GOLD = ROOT / 'data' / 'gold'
OUT = Path('outputs')
OUT.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(GOLD / 'fact_sales.csv', parse_dates=['InvoiceDate'] if False else None)
# flexible load — try common gold names
cands = list(GOLD.glob('*.csv')) + list(GOLD.glob('*.parquet'))
print('gold files:', [p.name for p in cands])


In [ ]:
# Prefer mart_kpi / fact if present
kpi_path = GOLD / 'mart_kpi.csv'
monthly_path = GOLD / 'mart_monthly.csv'
feat_path = GOLD / 'ml_customer_features.csv'

if monthly_path.exists():
    monthly = pd.read_csv(monthly_path)
    display(monthly.head())
    print('months', len(monthly), 'revenue_sum', round(monthly['Revenue'].sum(), 2) if 'Revenue' in monthly.columns else 'n/a')

if feat_path.exists():
    feat = pd.read_csv(feat_path)
    print('customers', len(feat), 'registered', int((feat['CustomerKey'] > 0).sum()) if 'CustomerKey' in feat.columns else 'n/a')
    if 'Churned90' in feat.columns:
        print('churn90_rate', round(feat.loc[feat['CustomerKey'] > 0, 'Churned90'].mean(), 3))


In [ ]:
if monthly_path.exists() and 'Revenue' in monthly.columns:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    x = monthly['YearMonth'] if 'YearMonth' in monthly.columns else monthly.index
    ax.plot(x, monthly['Revenue'] / 1e6, marker='o', markersize=3)
    ax.set_title('Monthly revenue (£M)')
    ax.tick_params(axis='x', labelrotation=45, labelsize=7)
    fig.tight_layout()
    fig.savefig(OUT / 'monthly_revenue_eda.png', dpi=120)
    plt.show()
